# 05 — Finite-shot SPSA with 1–6 faulty CNOT locations

This experiment begins with the six-parameter three-qubit circuit from notebook 04. It compares $R=1$ and $R=3$ rounds for every active-parameter count, and $R=5$ for counts 3, 4, and 6. Each shot encodes once, measures/resets ancillas between rounds, and records every syndrome and final data output. Only the **mean per-round nonzero-syndrome loss** trains Qiskit SPSA. No earlier two-gate exploratory optimization is used.

The ideal simulated gate parameters are all zero; a low syndrome loss can also arise at nonzero angles due to degenerate gate signatures. There is no injected physical error or active correction in this experiment.

In [ ]:
from pathlib import Path
import json
import sys
from time import perf_counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from qec_calibration import (
    SPSAConfig, build_repeated_round_template, calibrate_repeated_rounds,
    round_syndrome_loss, single_fault_response_rank, build_syndrome_template,
)

FAST_MODE = True  # Set False for the complete 45-run, 100-iteration experiment.
MAXITER = 15 if FAST_MODE else 100
TRAIN_SHOTS = 256 if FAST_MODE else 1024
VALIDATION_SHOTS = 1024 if FAST_MODE else 4096
TRIAL_COUNT = 1 if FAST_MODE else 3
ROUNDS_BY_SIZE = {1: (1, 3), 2: (1, 3), 3: (1, 3, 5),
                  4: (1, 3, 5), 5: (1, 3), 6: (1, 3, 5)}
ORDER = ("enc_0_1", "syn_0_left", "enc_0_2", "syn_0_right",
         "syn_1_left", "syn_1_right")
BASE = np.array([0.6, -0.5, 0.4, -0.35, 0.3, -0.45])
NOISE_SD = 0.04
START_SEED = 314

models = {r: build_repeated_round_template(3, rounds=r) for r in (1, 3, 5)}
assert len(ORDER) == len(BASE) == len(models[1].gate_ids)
assert set(ORDER) == set(models[1].gate_ids)
print("Training configuration:", {"maxiter": MAXITER, "shots": TRAIN_SHOTS,
       "validation_shots": VALIDATION_SHOTS, "trials": TRIAL_COUNT})
print("Response rank of isolated one-round fault signatures:",
      single_fault_response_rank(build_syndrome_template(3)))
print("CNOT operations per shot:", {r: 2 + 4*r for r in models})


## Reused starts and independent finite-shot runs

For each `(active_parameter_count, trial_id)`, draw small seeded perturbations of the specified starting angles **once** and reuse that exact vector at every $R$. Inactive locations stay exactly zero. SPSA uses 100 iterations, 1,024 training shots per objective evaluation, constant learning rate 0.7, and constant perturbation 0.12 in full mode. Initial/final validation uses fresh 4,096-shot runs; no validation data is supplied to SPSA. `FAST_MODE` makes a short functional check, not publishable results.

In the full experiment there are 18 $R=1$ runs, 18 $R=3$ runs, and nine $R=5$ runs. Trial ranges with only three repetitions are descriptive, not confidence intervals. The number of shots and iterations is fixed across $R$, while the number of CNOT operations per shot grows as $2+4R$.

In [ ]:
rng = np.random.default_rng(START_SEED)
starts = {}
for size in range(1, 7):
    for trial in range(TRIAL_COUNT):
        perturbation = rng.normal(0.0, NOISE_SD, size=size)
        starts[size, trial] = {name: float(value) for name, value in
                               zip(ORDER[:size], BASE[:size] + perturbation)}

results = {}
records = []
for size in range(1, 7):
    active = ORDER[:size]
    for trial in range(TRIAL_COUNT):
        for r in ROUNDS_BY_SIZE[size]:
            seed = 1000 + 100*size + trial  # matched across R for this trial
            config = SPSAConfig(maxiter=MAXITER, train_shots=TRAIN_SHOTS,
                                validation_shots=VALIDATION_SHOTS,
                                learning_rate=0.7, perturbation=0.12, seed=seed)
            started = perf_counter()
            result = calibrate_repeated_rounds(models[r], starts[size, trial],
                                               active_gate_ids=active, config=config)
            elapsed = perf_counter() - started
            results[r, size, trial] = result
            before = np.array([result.initial_angles[g] for g in active])
            after = np.array([result.optimized_angles[g] for g in active])
            wrapped = (after + np.pi) % (2*np.pi) - np.pi
            records.append(dict(rounds=r, active_parameters=size, trial=trial,
                                initial_loss=result.initial_loss, final_loss=result.final_loss,
                                initial_norm=float(np.linalg.norm(before)),
                                final_norm=float(np.linalg.norm(after)),
                                final_wrapped_norm=float(np.linalg.norm(wrapped)),
                                nfev=result.function_evaluations, runtime_seconds=elapsed,
                                seed=seed, train_shots=TRAIN_SHOTS,
                                validation_shots=VALIDATION_SHOTS, maxiter=MAXITER))
            print(f"R={r} m={size} trial={trial}: {result.initial_loss:.4f} → "
                  f"{result.final_loss:.4f}; ||theta||={np.linalg.norm(after):.3f}; "
                  f"nfev={result.function_evaluations}; {elapsed:.1f}s", flush=True)
trials = pd.DataFrame(records)
display(trials)


## Persist the complete validation records

The JSON contains every run's starting/final angles, SPSA trace, and complete joint syndrome-history/final-data counts from the independent initial and final validation batches. The optimization function sees only the average per-round syndrome loss. The JSON is generated when this cell executes; it is an experiment output rather than a prerequisite for the notebook.

In [ ]:
result_dir = ROOT / "notebooks" / "results"
result_dir.mkdir(parents=True, exist_ok=True)
result_path = result_dir / ("repeated_scaling_fast.json.gz" if FAST_MODE else "repeated_scaling_full.json.gz")

def pack_counts(counts):
    return [{"rounds": list(key[:-1]), "final_data": key[-1], "shots": int(count)}
            for key, count in sorted(counts.items())]

payload = {
    "configuration": {"fast_mode": FAST_MODE, "rounds_by_size": ROUNDS_BY_SIZE,
                      "order": ORDER, "base_angles": BASE.tolist(),
                      "start_seed": START_SEED, "start_noise_sd": NOISE_SD,
                      "maxiter": MAXITER, "train_shots": TRAIN_SHOTS,
                      "validation_shots": VALIDATION_SHOTS, "trials": TRIAL_COUNT,
                      "learning_rate": 0.7, "perturbation": 0.12,
                      "objective": "mean_per_round_nonzero_syndrome"},
    "runs": [{**record, "initial_angles": results[record["rounds"], record["active_parameters"], record["trial"]].initial_angles,
              "final_angles": results[record["rounds"], record["active_parameters"], record["trial"]].optimized_angles,
              "initial_counts": pack_counts(results[record["rounds"], record["active_parameters"], record["trial"]].initial_counts),
              "final_counts": pack_counts(results[record["rounds"], record["active_parameters"], record["trial"]].final_counts),
              "history": list(results[record["rounds"], record["active_parameters"], record["trial"]].history)}
             for record in records],
}
import gzip
with gzip.open(result_path, "wt", encoding="utf-8") as file:
    json.dump(payload, file)
print("Saved:", result_path, "bytes:", result_path.stat().st_size)


## Compare $R=1$ and $R=3$ across one through six active parameters

Both axes summarize independently validated initial and final values. Solid lines are medians across trials, shaded regions are trial minima to maxima. The left axis is the **per-round** syndrome loss. The right axis is the raw Euclidean angle error relative to zero, which is known only in simulation. The rank of isolated one-round fault signatures is three, but that rank alone does not guarantee or rule out identification by nonlinear multi-round statistics.

In [ ]:
summary = trials.groupby(["rounds", "active_parameters"], as_index=False).agg(
    initial_loss_med=("initial_loss", "median"), initial_loss_min=("initial_loss", "min"),
    initial_loss_max=("initial_loss", "max"), final_loss_med=("final_loss", "median"),
    final_loss_min=("final_loss", "min"), final_loss_max=("final_loss", "max"),
    initial_norm_med=("initial_norm", "median"), initial_norm_min=("initial_norm", "min"),
    initial_norm_max=("initial_norm", "max"), final_norm_med=("final_norm", "median"),
    final_norm_min=("final_norm", "min"), final_norm_max=("final_norm", "max"),
)
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, constrained_layout=True)
for row, r in enumerate((1, 3)):
    frame = summary[summary.rounds == r]
    x = frame.active_parameters.to_numpy()
    for col, stem, label in [(0, "loss", "Mean per-round nonzero syndrome"),
                              (1, "norm", "Raw angle norm (radians)")]:
        ax = axes[row, col]
        for status, color in [("initial", "tab:gray"), ("final", "tab:blue")]:
            ax.plot(x, frame[f"{status}_{stem}_med"].to_numpy(), marker="o", color=color,
                    label=f"{status.capitalize()} median")
            ax.fill_between(x, frame[f"{status}_{stem}_min"].to_numpy(),
                            frame[f"{status}_{stem}_max"].to_numpy(), color=color, alpha=0.18,
                            label=f"{status.capitalize()} trial range")
        ax.axvline(3.5, color="black", linestyle=":", alpha=0.55)
        ax.set(title=f"R={r}: {label}", ylabel=label, xticks=x)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
    axes[row, 0].set_xlabel("Active parameters")
    axes[row, 1].set_xlabel("Active parameters")
plt.show()
display(summary)


## Matched angle and sampled-loss traces: four and six active gates

For each size, select the $R=1$ trial nearest its median **final validation loss**, then display that same trial ID for $R=3$. The four/six angle lines trace the optimizer's complete parameter vector, not independent fits. A zero line marks the ideal parameters. The sampled objective is noisy; report final performance using the independent validation summary above. A final low loss with a large angle norm points to remaining degeneracy or an alternative low-loss solution.

In [ ]:
for size in (4, 6):
    group = trials[(trials.rounds == 1) & (trials.active_parameters == size)]
    median = group.final_loss.median()
    trial_id = int(group.loc[(group.final_loss - median).abs().idxmin(), "trial"])
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True,
                             gridspec_kw={"width_ratios": [2, 1]})
    for row, r in enumerate((1, 3)):
        result = results[r, size, trial_id]
        trace = pd.DataFrame(result.history)
        palette = plt.cm.tab10(np.linspace(0, 1, size))
        for i, gate in enumerate(ORDER[:size]):
            idx = models[r].gate_ids.index(gate)
            y = [angles[idx] for angles in trace.angles]
            axes[row, 0].plot(trace.iteration, y, label=gate, color=palette[i],
                              linestyle="-" if gate.startswith("enc") else "--")
        axes[row, 0].axhline(0, color="black", linestyle=":", linewidth=1)
        axes[row, 0].set(title=f"{size} active, R={r}: angles (trial {trial_id})",
                         xlabel="SPSA iteration", ylabel="Angle (radians)")
        axes[row, 0].legend(fontsize=8, ncol=2)
        axes[row, 1].plot(trace.iteration, trace.loss, color="tab:purple")
        axes[row, 1].set(title=f"R={r}: sampled objective", xlabel="SPSA iteration",
                         ylabel="Mean per-round syndrome loss")
        for ax in axes[row]:
            ax.grid(alpha=0.25)
    plt.show()


## Focused $R=3$ versus $R=5$ comparison

Compare matched starting vectors and trial IDs for sizes 3, 4, 6. The two outcomes are final **validation** loss and angle norm. Each $R=5$ shot executes 22 CNOTs versus 14 at $R=3$, so equal shot budgets do not imply equal gate budget. Trial ranges are descriptive.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for ax, metric, ylabel in zip(axes, ("final_loss", "final_norm"),
                              ("Final per-round syndrome loss", "Final raw angle norm")):
    for r, color in ((3, "tab:blue"), (5, "tab:orange")):
        x = summary[(summary.rounds == r) & summary.active_parameters.isin((3, 4, 6))]
        size = x.active_parameters.to_numpy()
        ax.plot(size, x[f"{metric}_med"].to_numpy(), marker="o", label=f"R={r}", color=color)
        ax.fill_between(size, x[f"{metric}_min"].to_numpy(),
                        x[f"{metric}_max"].to_numpy(), alpha=0.2, color=color)
    ax.set(xlabel="Active parameters", ylabel=ylabel, xticks=[3, 4, 6])
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()


## Descriptive history analysis, outside the objective

For matched six-gate runs, inspect the per-round probability of each syndrome, the fraction of adjacent rounds with identical syndrome, and final data words conditional on a nonzero syndrome anywhere. These use **validation counts only**. A persistent syndrome can suggest a data branch, while transient outcomes can suggest ancilla measurement errors; with simultaneous faults the interpretation is less direct. The final data word measures the data basis state, not a decoded logical failure rate.

In [ ]:
size = 6
# Match the R=1 trial nearest its median final validation loss.
group = trials[(trials.rounds == 1) & (trials.active_parameters == size)]
selected_trial = int(group.loc[(group.final_loss - group.final_loss.median()).abs().idxmin(), "trial"])
rows = []
for r in (1, 3, 5):
    counts = results[r, size, selected_trial].final_counts
    shots = sum(counts.values())
    for t in range(r):
        for syndrome in ("00", "10", "11", "01"):
            p = sum(c for key, c in counts.items() if key[t] == syndrome) / shots
            rows.append({"R": r, "round": t, "syndrome": syndrome, "probability": p})
    if r > 1:
        persistence = [sum(c for key, c in counts.items() if key[t] == key[t-1]) / shots
                       for t in range(1, r)]
        print(f"R={r}, equal adjacent syndrome rates:", np.round(persistence, 3))
    nonzero = {word: sum(c for key, c in counts.items()
                         if key[-1] == word and any(s != "00" for s in key[:-1]))
               for word in ("000", "001", "010", "011", "100", "101", "110", "111")}
    print(f"R={r}, final data conditional on any nonzero round:",
          {word: count for word, count in nonzero.items() if count})
pattern_table = pd.DataFrame(rows)
display(pattern_table.pivot_table(index=["R", "round"], columns="syndrome", values="probability"))


## Interpretation

Compare validated loss with raw and modulo-$2\pi$ wrapped angle norms: an improved syndrome objective need not identify every individual gate. Repeated rounds can reveal temporal structure in the stored records, while same-ancilla target rotations that cancel remain exactly indistinguishable for this preparation and readout. This experiment has no recovery and does not establish logical error-correction performance. `FAST_MODE` is solely a functional check; switch it off for the specified full sweep.